In [16]:
import jupyter_black
jupyter_black.load()

import pandas as pd

from Levenshtein import ratio

from ccl_science_data.common import load_map, EntC, GenReader, iter_dfs, parse_id, get_arr, oa_root
from tqdm.notebook import tqdm

In [4]:
to_drop = [
    "SOCCER PLAYER",
    "ACTOR",
    "SINGER",
    "ATHLETE",
    "MUSICIAN",
    "FILM DIRECTOR",
    "MILITARY PERSONNEL",
    "BASKETBALL PLAYER",
    "CYCLIST",
    "TENNIS PLAYER",
    "RACING DRIVER",
    "WRESTLER",
    "SKIER",
    "SWIMMER",
    "HOCKEY PLAYER",
    "COACH",
    "CHESS PLAYER",
    "BOXER",
    "SKATER",
    "HANDBALL PLAYER",
]

df = pd.concat(
    _df.loc[~_df["occupation"].isin(to_drop) & ~(_df["deathyear"].astype(float) < 1950), :]
    for _df in pd.read_csv(
        "~/Downloads/person_2020_update.csv.bz2", chunksize=1_000_000
    )
)

/tmp/ipykernel_1517865/2115394758.py:24: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat(


In [5]:
e = EntC.AUTHORS
step = "derive_links5"

In [6]:
sizet = 8
adir = f"{step}/{e}-semantic-ids"
ss = 0
sids = []
ftxt = (oa_root /adir /"targets").read_bytes()
for s in map(int, get_arr(f"{adir}/sizes", sizet)):
    sids.append(ftxt[ss : (ss + s)].decode())
    ss = ss + s
sids = sids[:-1]

In [9]:
pd.read_csv('s3://tmp-borza-public-cyx/oa-to-wiki-authors.csv.gz')

,rl_i,slug,oa_id
0,5,Michael_Grätzel,5109160404
1,2190932,Geoffrey_Hinton,5108093963
2,2190937,Robert_Weinberg,5012557712
3,372694,George_M._Whitesides,5051537916
4,1806632,Omar_M._Yaghi,5044268160
...,...,...,...
111,965,Roderick_MacKinnon,5025045433
112,2738304,Hilary_Koprowski,5050328478
113,812,Arieh_Warshel,5088665303
114,1391639,Robert_H._MacArthur,5082781942


In [11]:
ent = EntC.AUTHORS

In [12]:
wdf = pd.concat(_df.dropna() for _df in tqdm(iter_dfs(ent, "ids", cols=["openalex", "wikipedia"])))

0it [00:00, ?it/s]

/home/borza/science-data/ccl_science_data/common.py:181: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for _df in pd.read_csv(get_csv_path(ent, sub), chunksize=chunk, usecols=cols):
/home/borza/science-data/ccl_science_data/common.py:181: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for _df in pd.read_csv(get_csv_path(ent, sub), chunksize=chunk, usecols=cols):
/home/borza/science-data/ccl_science_data/common.py:181: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for _df in pd.read_csv(get_csv_path(ent, sub), chunksize=chunk, usecols=cols):
/home/borza/science-data/ccl_science_data/common.py:181: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  for _df in pd.read_csv(get_csv_path(ent, sub), chunksize=chunk, usecols=cols):
/home/borza/science-data/ccl_science_data/common

In [13]:
emap = load_map(ent)

In [14]:
print("".join(map(lambda s: f"- {s}\n", wdf["wikipedia"].tolist())))

- http://en.wikipedia.org/wiki/Josiah_Carberry
- https://es.wikipedia.org/wiki/Mauricio_Toro_Berm%C3%BAdez
- https://en.wikipedia.org/wiki/Christopher_Cs%C3%ADkszentmih%C3%A1lyi
- https://en.wikipedia.org/wiki/Alan_Pilkington
- https://en.wikipedia.org/wiki/Brigitte_Servatius
- https://de.wikipedia.org/wiki/Sebastian_Kempgen
- https://en.wikipedia.org/wiki/Monika_Ritsch-Marte
- https://eu.wikipedia.org/wiki/Iker_Gonz%C3%A1lez-Allende
- https://en.wikipedia.org/wiki/Ihor_Yukhnovskyi



In [17]:
gr = GenReader("..")

In [19]:
adf = (
    pd.DataFrame(
        {
            "name": gr.get_names(EntC.AUTHORS),
            "cc": gr.get_ccounts(EntC.AUTHORS)[:-1],
            "sid": sids,
        }
    )
    .loc[lambda df: df["cc"] > 40_000]
    .sort_values("cc", ascending=False)
    .assign(rl_url=lambda df: "https://www.rankless.org/authors/" + df["sid"])
)

In [20]:
fzdf = pd.concat(
    df.loc[lambda df: df["name"].apply(lambda e: ratio(e, r["name"]) > 0.9)].assign(
        rl_i=i
    )
    for i, r in tqdm(adf.drop_duplicates("name").iterrows())
)

0it [00:00, ?it/s]

In [21]:
o_fdf = (
    fzdf.loc[
        :,
        [
            "id",
            "wd_id",
            "wp_id",
            "slug",
            "name",
            "occupation",
            "birthyear",
            "deathyear",
            "rl_i",
        ],
    ]
    .merge(adf, left_on="rl_i", right_index=True)
    .assign(wiki_url=lambda df: "https://en.wikipedia.org/wiki/" + df["slug"])
)

In [22]:
print(
    "".join(
        f"{e1} -  {e2} ({e3})\n"
        for (e1, e2, e3) in o_fdf.sort_values("occupation")
        .loc[:, ["wiki_url", "rl_url", "occupation"]]
        .values
    )
)

https://en.wikipedia.org/wiki/David_H._Levy -  https://www.rankless.org/authors/david-e-levy (ASTRONOMER)
https://en.wikipedia.org/wiki/Michael_E._Brown -  https://www.rankless.org/authors/michael-s-brown (ASTRONOMER)
https://en.wikipedia.org/wiki/Elizabeth_Blackburn -  https://www.rankless.org/authors/elizabeth-h-blackburn (BIOLOGIST)
https://en.wikipedia.org/wiki/Judah_Folkman -  https://www.rankless.org/authors/judah-folkman (BIOLOGIST)
https://en.wikipedia.org/wiki/Bert_Sakmann -  https://www.rankless.org/authors/bert-sakmann (BIOLOGIST)
https://en.wikipedia.org/wiki/Oliver_Smithies -  https://www.rankless.org/authors/oliver-smithies (BIOLOGIST)
https://en.wikipedia.org/wiki/Erwin_Neher -  https://www.rankless.org/authors/erwin-neher (BIOLOGIST)
https://en.wikipedia.org/wiki/Robert_Sapolsky -  https://www.rankless.org/authors/robert-m-sapolsky (BIOLOGIST)
https://en.wikipedia.org/wiki/Gerald_Edelman -  https://www.rankless.org/authors/gerald-m-edelman (BIOLOGIST)
https://en.wikiped

In [23]:
misses = [
    "https://www.rankless.org/authors/david-e-levy",
    "https://www.rankless.org/authors/michael-s-brown",
    "https://www.rankless.org/authors/wolf-singer",
    "https://www.rankless.org/authors/david-richardson",
    "https://www.rankless.org/authors/albert-hofman",
    "https://www.rankless.org/authors/james-hone",
    "https://www.rankless.org/authors/karel-svoboda",
    "https://www.rankless.org/authors/michael-f-clarke",
    "https://www.rankless.org/authors/r-stanley-williams",
    "https://www.rankless.org/authors/michael-p-manns",
    "https://www.rankless.org/authors/john-robertson",
    "https://www.rankless.org/authors/yang-zhang",
    "https://www.rankless.org/authors/john-f-thompson",
    "https://www.rankless.org/authors/helen-christensen",
    "https://www.rankless.org/authors/robert-b-jackson",
    "https://www.rankless.org/authors/michael-r-hayden",
    "https://www.rankless.org/authors/david-l-paterson",
    "https://www.rankless.org/authors/garret-a-fitzgerald",
    "https://www.rankless.org/authors/chen-chen-2",
    "https://www.rankless.org/authors/bryan-williams",
    "https://www.rankless.org/authors/giacomo-rizzolatti-2",
    "https://www.rankless.org/authors/francesco-montorsi",
    "https://www.rankless.org/authors/matthew-stephens",
    "https://www.rankless.org/authors/erik-lindahl",
    "https://www.rankless.org/authors/anthony-howell",
    "https://www.rankless.org/authors/james-j-collins",
    "https://www.rankless.org/authors/john-y-campbell",
    "https://www.rankless.org/authors/w-d-hamilton-2",  # TODO
    "https://www.rankless.org/authors/craig-b-thompson",
    "https://www.rankless.org/authors/j-michael-bishop",
]

In [117]:
o_fdf.set_index("rl_url").drop(misses).to_csv("~/Downloads/rl-wiki-matches.csv")

In [24]:
match_df = (
    o_fdf.set_index("rl_url")
    .drop(misses)
    .set_index("rl_i")
    .assign(oa_id=pd.Series({v: k for k, v in emap.items()}))
)

In [126]:
match_df.loc[:, ["slug", "oa_id"]].to_csv("~/Downloads/oa-to-wiki-authors.csv.gz")

In [77]:
fzdf["occupation"].value_counts()#["rl_i"].nunique()

occupation
CHEMIST                33
BIOLOGIST              30
PHYSICIST              13
PHYSICIAN              12
ECONOMIST              10
PSYCHOLOGIST            8
POLITICIAN              5
SOCIOLOGIST             4
COMPUTER SCIENTIST      4
WRITER                  4
MATHEMATICIAN           3
POLITICAL SCIENTIST     2
COMPOSER                2
ASTRONOMER              2
RELIGIOUS FIGURE        2
PHILOSOPHER             2
COMIC ARTIST            1
SNOOKER                 1
JUDGE                   1
PRESENTER               1
TABLE TENNIS PLAYER     1
GEOLOGIST               1
BUSINESSPERSON          1
EXTREMIST               1
ENGINEER                1
CRICKETER               1
MODEL                   1
Name: count, dtype: int64

In [57]:
mdf = adf.merge(df)

In [58]:
mdf["occupation"].value_counts()

occupation
CHEMIST                25
BIOLOGIST              23
PHYSICIST              11
PHYSICIAN               9
PSYCHOLOGIST            6
ECONOMIST               5
COMPUTER SCIENTIST      3
SOCIOLOGIST             3
POLITICAL SCIENTIST     2
PHILOSOPHER             2
RELIGIOUS FIGURE        1
ENGINEER                1
COMPOSER                1
Name: count, dtype: int64

In [55]:
mdf.loc[lambda df: df["occupation"].isin(["COMPOSER", "RELIGIOUS FIGURE"])]

,name,cc,id,wd_id,wp_id,slug,occupation,prob_ratio,gender,twitter,alive,l,hpi_raw,bplace_name,bplace_lat,bplace_lon,bplace_geonameid,bplace_country,birthdate,birthyear,dplace_name,dplace_lat,dplace_lon,dplace_geonameid,dplace_country,deathdate,deathyear,bplace_geacron_name,dplace_geacron_name,is_group,l_,age,non_en_page_views,coefficient_of_variation,hpi
62,Hirotugu Akaike,47153,4291862,Q2559832,4291862,Hirotugu_Akaike,RELIGIOUS FIGURE,1.341727,F,NaN,False,20,20,"Fujinomiya, Shizuoka",35.222111,138.621611,328991.0,Japan,1927-11-05,1927.0,Ibaraki Prefecture,36.233333,140.283333,181029.0,Japan,2009-08-04,2009.0,NaN,NaN,False,3.689702,93.0,5780.0,2.701078,59.306662
72,Karel Svoboda,44321,22728486,Q356687,22728486,Karel_Svoboda_(composer),COMPOSER,13.192315,M,NaN,False,22,22,Prague,50.083333,14.416667,23844.0,Czechia,1938-12-19,1938.0,Jevany,49.966667,14.816667,23583479.0,Czechia,2007-01-28,2007.0,Czechoslovakia,NaN,False,4.005634,82.0,44838.0,2.661908,65.074659
